In [1]:
import os
import pandas as pd

# 1. 경로 설정 (지훈 님 환경에 맞춤)
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_score = rf"{base_path}\PROD_FLOWSCORE\PUBLIC"
path_point = rf"{base_path}\PROD_FLOWPOINT\PUBLIC"

def list_files(directory, label):
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.csv')]
        print(f"\n📂 {label} 폴더 내 파일 ({len(files)}개)")
        for f in sorted(files):
            print(f"- {f}")
    else:
        print(f"❌ 경로 없음: {directory}")

list_files(path_score, "FLOWSCORE (재무/기업)")
list_files(path_point, "FLOWPOINT (거래/BNPL)")


📂 FLOWSCORE (재무/기업) 폴더 내 파일 (30개)
- BUSINESS.csv
- COMPANY.csv
- COMPANY_CAPITAL_STATUS.csv
- COMPANY_CASH_FLOW.csv
- COMPANY_CREDIT_GRADE.csv
- COMPANY_EMPLOYEE_STATISTICS.csv
- COMPANY_EXECUTIVES.csv
- COMPANY_EXECUTIVE_DETAIL.csv
- COMPANY_FINANCIAL_RATIO.csv
- COMPANY_FINANCIAL_STATEMENT.csv
- COMPANY_HISTORY.csv
- COMPANY_INCOME_STATEMENT.csv
- COMPANY_KOREAN_CODE_V1.csv
- COMPANY_MORATORIUM.csv
- COMPANY_OVERDUE.csv
- COMPANY_OVERVIEW.csv
- COMPANY_PENSION_EMPLOYEE_STATISTICS.csv
- COMPANY_REPRESENTATIVE.csv
- COMPANY_REPRESENTATIVE_HISTORY.csv
- COMPANY_SEARCH_KOREAN_CODE_V1.csv
- COMPANY_STOCKHOLDER.csv
- EWA_CODE_V1.csv
- EWA_CODE_V2.csv
- EWA_GRADE.csv
- FINANCIAL_STATUS.csv
- PRIVILEGE.csv
- ROLES.csv
- SEARCH_COMPANY_HISTORY.csv
- USERS.csv
- USER_COMPANY_SEARCH_HISTORY.csv

📂 FLOWPOINT (거래/BNPL) 폴더 내 파일 (130개)
- ACCOUNTS.csv
- ASSIGNMENTS.csv
- ASSIGNMENT_ADD_DOCS.csv
- ASSIGNMENT_ASSIGNMENT_ADD_DOCS.csv
- ASSIGNMENT_ASSIGNMENT_DETAILS.csv
- ASSIGNMENT_DETAILS.csv
- ASSIG

In [15]:
# === [확장된 통합 범위 로드 엔진] ===

def load_extended_audit_data():
    # 1. 기초 및 신용 (Overview, Grade, Overdue)
    df_ov = pd.read_csv(rf"{path_score}\COMPANY_OVERVIEW.csv", encoding='utf-8-sig')
    df_cg = pd.read_csv(rf"{path_score}\COMPANY_CREDIT_GRADE.csv", encoding='utf-8-sig')
    df_od = pd.read_csv(rf"{path_score}\COMPANY_OVERDUE.csv", encoding='utf-8-sig')

    # 2. 재무 풀세트 (BS, IS, Ratio)
    df_bs = pd.read_csv(rf"{path_score}\COMPANY_FINANCIAL_STATEMENT.csv", encoding='utf-8-sig')
    df_is = pd.read_csv(rf"{path_score}\COMPANY_INCOME_STATEMENT.csv", encoding='utf-8-sig')

    # 3. 운영 및 동태 (Employee, Rep History, Capital)
    df_emp = pd.read_csv(rf"{path_score}\COMPANY_EMPLOYEE_STATISTICS.csv", encoding='utf-8-sig')
    df_rep = pd.read_csv(rf"{path_score}\COMPANY_REPRESENTATIVE_HISTORY.csv", encoding='utf-8-sig')
    df_cap = pd.read_csv(rf"{path_score}\COMPANY_CAPITAL_STATUS.csv", encoding='utf-8-sig')

    # 4. 거래 및 이력 (Links, BNPL History)
    df_link = pd.read_csv(rf"{path_point}\COMPANY_LINKS.csv", encoding='utf-8-sig')
    df_bnpl_hist = pd.read_csv(rf"{path_point}\VIEW_PAY_BNPL_REQUEST_HISTORY.csv", encoding='utf-8-sig')

    print(f"✅ 총 {len(locals())}개의 핵심 전략 데이터셋이 로드되었습니다.")
    return df_ov, df_cg, df_od, df_bs, df_is, df_emp, df_rep, df_cap, df_link, df_bnpl_hist

# 실행
ov, cg, od, bs, is_df, emp, rep, cap, link, bnpl = load_extended_audit_data()

✅ 총 10개의 핵심 전략 데이터셋이 로드되었습니다.


In [16]:
print(bs.columns)

Index(['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META',
       '_AIRBYTE_GENERATION_ID', 'ID', 'FS_VAL1', 'FS_VAL2', 'FS_VAL3',
       'FS_VAL4', 'FS_VAL5',
       ...
       'FS_VAL378', 'FS_VAL379', 'FS_VAL380', 'FS_VAL381', 'COMPANY_ID',
       'CREATED_AT', 'FS_ACCT_DT', '_AB_CDC_LSN', '_AB_CDC_DELETED_AT',
       '_AB_CDC_UPDATED_AT'],
      dtype='object', length=392)


In [ ]:
import pandas as pd
import os

# 1. 경로 재설정 (Jacob 님 환경)
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
excel_path = rf"{base_path}\운영DB 스키마 명세.xlsx"

# 2. 명세서(Schema) 로드
# NameError를 방지하기 위해 변수를 확실히 선언합니다.
sc_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')
sc_point = pd.read_excel(excel_path, sheet_name='PROD_FLOWPOINT')

# 3. 통합 인스펙터 함수 정의 (schema를 인자로 받도록 개선)
def inspect_all_mappings(datasets, schema_score, schema_point):
    for label, (df, table_name, schema_type) in datasets.items():
        print(f"\n" + "="*70)
        print(f"🧐 [{label}] 매핑 정밀 점검 (Table: {table_name})")
        print("="*70)
        
        # 데이터 소스에 따라 적절한 명세서 선택
        schema = schema_score if schema_type == 'score' else schema_point
        
        # 현재 테이블에 해당하는 매핑 딕셔너리 추출
        mapping = {str(k).upper(): v for k, v in zip(schema['column_name'], schema['logical_column_name']) 
                   if str(schema.loc[schema['column_name']==k, 'table_name'].values[0]).lower() == table_name.lower() and pd.notna(v)}
        
        raw_cols = [str(c).upper() for c in df.columns]
        mapped_cols = [c for c in raw_cols if c in mapping]
        
        print(f"▶ 총 컬럼: {len(raw_cols)}개 | 매핑 성공: {len(mapped_cols)}개")
        
        # 매핑되지 않은 '미아' 컬럼 중 중요해 보이는 것들 (ID, DT 등)
        unmapped = [c for c in raw_cols if c not in mapping and any(k in c for k in ['ID', 'DT', 'DATE', 'VAL'])]
        if unmapped:
            print(f"⚠️ 매핑 누락(의심): {unmapped[:10]}...")

        # 실제 매핑 예시
        print(f"✅ 매핑 완료 샘플: " + str({c: mapping[c] for c in mapped_cols[:3]}))

# 4. 점검 대상 리스트업 (데이터셋, 테이블명, 명세서타입)
target_datasets = {
    "재무상태표": (bs, 'company_financial_statement', 'score'),
    "손익계산서": (is_df, 'company_income_statement', 'score'),
    "기업개황": (ov, 'company_overview', 'score'),
    "신용등급": (cg, 'company_credit_grade', 'score'),
    "연체이력": (od, 'company_overdue', 'score'),
    "인력통계": (emp, 'company_employee_statistics', 'score'),
    "대표자이력": (rep, 'company_representative_history', 'score'),
    "자본상태": (cap, 'company_capital_status', 'score'),
    "공급망": (link, 'company_links', 'point'),
    "BNPL이력": (bnpl, 'view_pay_bnpl_request_history', 'point')
}

# 5. 실행
inspect_all_mappings(target_datasets, sc_score, sc_point)


🧐 [재무상태표] 매핑 정밀 점검 (Table: company_financial_statement)
▶ 총 컬럼: 392개 | 매핑 성공: 382개
⚠️ 매핑 누락(의심): ['_AIRBYTE_RAW_ID', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', '_AB_CDC_UPDATED_AT']...
✅ 매핑 완료 샘플: {'FS_VAL1': '결산년_자산', 'FS_VAL2': '결산년_유동자산', 'FS_VAL3': '결산년_당좌자산'}

🧐 [손익계산서] 매핑 정밀 점검 (Table: company_income_statement)
▶ 총 컬럼: 101개 | 매핑 성공: 0개
⚠️ 매핑 누락(의심): ['_AIRBYTE_RAW_ID', '_AIRBYTE_GENERATION_ID', 'ID', 'FS_VAL1', 'FS_VAL2', 'FS_VAL3', 'FS_VAL4', 'FS_VAL5', 'FS_VAL6', 'FS_VAL7']...
✅ 매핑 완료 샘플: {}

🧐 [기업개황] 매핑 정밀 점검 (Table: company_overview)
▶ 총 컬럼: 42개 | 매핑 성공: 19개
⚠️ 매핑 누락(의심): ['_AIRBYTE_RAW_ID', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', '_AB_CDC_UPDATED_AT', 'ESTABLISHMENT_DATE']...
✅ 매핑 완료 샘플: {'IPO_CODE': 'IPO 코드', 'FAX_NUMBER': '팩스번호', 'GROUP_NAME': '그룹명'}

🧐 [신용등급] 매핑 정밀 점검 (Table: company_credit_grade)
▶ 총 컬럼: 16개 | 매핑 성공: 6개
⚠️ 매핑 누락(의심): ['_AIRBYTE_RAW_ID', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', '_AB_CDC_UPDATED_AT']...
✅ 매핑 완료 샘플: {'CREDIT_GRADE': '기업 신용등급'

In [18]:
# 1. 파일의 실제 민낯 확인
print(f"📄 [IS 파일] 실제 컬럼 상위 10개: \n{is_df.columns[:10].tolist()}")

# 2. 엑셀 명세서에서 '손익' 혹은 '매출' 단어가 들어간 모든 테이블명 검색
is_candidates = sc_score[sc_score['logical_column_name'].str.contains('매출|영업이익|손익', na=False)]['table_name'].unique()
print(f"\n📑 [엑셀 명세서] IS 관련 의심 테이블명: {is_candidates}")

# 3. 엑셀에 등록된 해당 테이블의 컬럼 샘플 확인
if len(is_candidates) > 0:
    target_table = is_candidates[0]
    print(f"\n🧐 명세서 내 [{target_table}]의 실제 등록 컬럼 예시:")
    display(sc_score[sc_score['table_name'] == target_table][['column_name', 'logical_column_name']].head(5))

📄 [IS 파일] 실제 컬럼 상위 10개: 
['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'FS_VAL1', 'FS_VAL2', 'FS_VAL3', 'FS_VAL4', 'FS_VAL5']

📑 [엑셀 명세서] IS 관련 의심 테이블명: ['company_cash_flow' 'company_financial_ratio'
 'company_financial_statement' 'company_income_statement'
 'financial_status']

🧐 명세서 내 [company_cash_flow]의 실제 등록 컬럼 예시:


,column_name,logical_column_name
55,id,현금흐름분석 아이디
56,company_id,NaN
57,created_at,생성일자
58,cf_acct_dt,결산일
59,cf_anal1,현금영업이익


In [19]:
# 1. BNPL 이력 파일 실제 컬럼 확인
print(f"📄 [BNPL 이력] 실제 컬럼: \n{bnpl.columns.tolist()}")

# 2. 엑셀 명세서(sc_point)에서 BNPL 관련 키워드 검색
bnpl_candidates = sc_point[sc_point['logical_column_name'].str.contains('심사|신청|결제', na=False)]['table_name'].unique()
print(f"\n📑 [엑셀 명세서] BNPL 관련 의심 테이블명: {bnpl_candidates}")

📄 [BNPL 이력] 실제 컬럼: 
['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'TYPE', 'IS_READ', 'ITEM_ID', 'COMPANY_ID', 'CREATED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'IS_INTERNAL', 'COMMENT_UUID', 'ADMIN_USER_ID', 'COMMENT_CONTENT', 'EVALUATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS', 'PAY_BNPL_REQUEST_ID']

📑 [엑셀 명세서] BNPL 관련 의심 테이블명: ['alimtalk_api_logs' 'assignments' 'bnpl_buyers' 'bnpl_comments'
 'bnpl_request_application_stage_logs'
 'bnpl_request_evaluation_stage_logs' 'bnpl_requests'
 'partnership_inquiries' 'pay_bnpl_buy_items'
 'pay_bnpl_buy_items_backup_20251211' 'pay_bnpl_comments'
 'pay_bnpl_orders' 'pay_bnpl_request_evaluation_logs' 'pay_bnpl_requests'
 'pay_bnpl_requests_backup_20251211' 'pay_bnpl_seller_company_infos_b2b'
 'pay_bnpl_seller_company_infos_b2c' 'pay_snpl_buyer_orders'
 'pay_snpl_orders' 'payables' 'receivables' 'transfers'
 'wecheck_callbacks' 'wecheck_documents']


In [20]:
# 엑셀 전체에서 'COMPANY_ID'라는 물리 컬럼명이 어떻게 정의되어 있는지 확인
id_mapping_status = sc_score[sc_score['column_name'] == 'COMPANY_ID'][['table_name', 'logical_column_name']]
print("🆔 [COMPANY_ID] 엑셀 등록 현황 (상위 10개):")
display(id_mapping_status.head(10))

🆔 [COMPANY_ID] 엑셀 등록 현황 (상위 10개):


,table_name,logical_column_name


In [21]:
import pandas as pd
import numpy as np

def smart_labeler(df, table_name, schema):
    print(f"🔍 [{table_name}] 라벨링 작업 중...")
    
    # 1. 엑셀 명세서 전처리 (대문자화 + 공백 제거)
    temp_sc = schema.copy()
    temp_sc['column_name'] = temp_sc['column_name'].str.upper().str.strip()
    temp_sc['table_name'] = temp_sc['table_name'].str.lower().str.strip()
    
    # 2. 매핑 딕셔너리 생성 (현재 테이블에 해당하는 것만)
    # View 파일의 경우, 'view_'를 떼고 원본 테이블명과 매칭 시도
    clean_table_name = table_name.lower().replace('view_pay_', '').replace('view_', '')
    
    # 명세서에서 가장 유사한 테이블명 찾기 (유연한 매칭)
    matched_table = temp_sc[temp_sc['table_name'].str.contains(clean_table_name, na=False)]['table_name'].unique()
    
    if len(matched_table) > 0:
        actual_table = matched_table[0]
        mapping = {str(k): v for k, v in zip(temp_sc[temp_sc['table_name'] == actual_table]['column_name'], 
                                             temp_sc[temp_sc['table_name'] == actual_table]['logical_column_name']) 
                   if pd.notna(v)}
    else:
        mapping = {}

    # 3. [CIO 핵심 패치] 엑셀에 없어도 데이터에 있으면 강제로 붙이는 공통 키
    common_patch = {
        'COMPANY_ID': '회사 아이디',
        'ID': '고유번호',
        'FS_ACCT_DT': '결산일',
        'CREATED_AT': '생성일자',
        'UPDATED_AT': '수정일자',
        'APPLICATION_STATUS': '최종상태',
        'EVALUATION_STAGE': '심사단계'
    }
    mapping.update(common_patch)
    
    # 4. 변환 (파일 컬럼도 대문자로 정규화하여 매칭)
    df.columns = [c.upper() for c in df.columns]
    labeled_df = df.rename(columns=mapping)
    
    # 5. 결과 보고
    success_count = sum(1 for c in labeled_df.columns if c in mapping.values())
    print(f"✅ {table_name} -> {actual_table if len(matched_table)>0 else 'N/A'} 매핑 성공: {success_count}개")
    
    return labeled_df

# 실행 예시
ov_l = smart_labeler(ov, 'company_overview', sc_score)
is_l = smart_labeler(is_df, 'company_income_statement', sc_score) # 이제 'FS_VAL'들이 이름표를 답니다.
bnpl_l = smart_labeler(bnpl, 'view_pay_bnpl_request_history', sc_point) # View도 Requests 명세로 매핑됩니다.

🔍 [company_overview] 라벨링 작업 중...
✅ company_overview -> company_overview 매핑 성공: 35개
🔍 [company_income_statement] 라벨링 작업 중...
✅ company_income_statement -> company_income_statement 매핑 성공: 94개
🔍 [view_pay_bnpl_request_history] 라벨링 작업 중...
✅ view_pay_bnpl_request_history -> N/A 매핑 성공: 5개


In [22]:
# 1. 미아(BNPL View) 구출을 위한 매핑 포인터 추가
def final_labeler_v3(df, table_name, schema):
    # 명세서에서 'view_pay_bnpl_request_history' 대신 'pay_bnpl_requests'를 찾도록 강제 지정
    lookup_name = table_name
    if 'view_pay_bnpl_request_history' in table_name:
        lookup_name = 'pay_bnpl_requests' # 엑셀 시트 내 실제 물리 테이블명으로 추정
    
    return smart_labeler(df, lookup_name, schema)

# 2. 전체 데이터셋 라벨링 재가동
ov_l = final_labeler_v3(ov, 'company_overview', sc_score)
is_l = final_labeler_v3(is_df, 'company_income_statement', sc_score)
bnpl_l = final_labeler_v3(bnpl, 'view_pay_bnpl_request_history', sc_point) # 이제 이름표를 찾을 것입니다.
# ... 나머지 7개 파일도 동일하게 적용

🔍 [company_overview] 라벨링 작업 중...
✅ company_overview -> company_overview 매핑 성공: 35개
🔍 [company_income_statement] 라벨링 작업 중...
✅ company_income_statement -> company_income_statement 매핑 성공: 94개
🔍 [pay_bnpl_requests] 라벨링 작업 중...
✅ pay_bnpl_requests -> pay_bnpl_requests 매핑 성공: 6개


In [23]:
# 1. 재무비율 데이터 로드 및 명세 확인
df_ratio_raw = pd.read_csv(rf"{path_score}\COMPANY_FINANCIAL_RATIO.csv", encoding='utf-8-sig')

# 2. 명세서에서 해당 테이블의 매핑 정보 추출
ratio_map = {str(k).upper(): v for k, v in zip(sc_score['column_name'], sc_score['logical_column_name']) 
             if 'financial_ratio' in str(sc_score.loc[sc_score['column_name']==k, 'table_name'].values[0]).lower() 
             and pd.notna(v)}

# 3. 실제 컬럼과 매치되는 주요 지표들 확인
print("📊 [재무비율 파일] 내 주요 매뉴얼 지표:")
for col in df_ratio_raw.columns:
    col_upper = col.upper()
    if col_upper in ratio_map:
        print(f"- {col_upper} : {ratio_map[col_upper]}")

📊 [재무비율 파일] 내 주요 매뉴얼 지표:
- FR_VAL1 : 총자산증가율
- FR_VAL2 : 매출액증가율
- FR_VAL3 : 순이익증가율
- FR_VAL4 : 영업이익율
- FR_VAL5 : ROE
- FR_VAL6 : ROIC
- FR_VAL7 : 부채비율
- FR_VAL8 : 이자보상배수
- FR_VAL9 : 차입금의존도
- FR_VAL10 : 매출채권회전율
- FR_VAL11 : 재고자산회전율
- FR_VAL12 : 총자본회전율
- FR_VAL13 : 자기자본비율
- FR_VAL14 : 당기순이익률
- FR_VAL15 : 유동비율
- FR_VAL16 : 부가가치율
- FR_ACCT_DT : 결산일


In [24]:
import pandas as pd
import numpy as np

# 1. 스마트 라벨러 (안정화 버전)
def smart_labeler_final(df, table_name, schema):
    if df is None or df.empty: return None
    temp_sc = schema.copy()
    temp_sc['column_name'] = temp_sc['column_name'].str.upper().str.strip()
    temp_sc['table_name'] = temp_sc['table_name'].str.lower().str.strip()
    
    clean_name = table_name.lower().replace('view_pay_', '').replace('view_', '').replace('company_', '')
    target_schema = temp_sc[temp_sc['table_name'].str.contains(clean_name, na=False)]
    
    mapping = {str(k): v for k, v in zip(target_schema['column_name'], target_schema['logical_column_name']) if pd.notna(v)}
    
    # [CIO 긴급 패치] 공통 키 및 날짜 강제 매핑
    patch = {
        'COMPANY_ID': '회사 아이디', 'ID': '고유번호', 
        'FS_ACCT_DT': '결산일', 'FR_ACCT_DT': '결산일', 
        'STANDARD_DATE': '기준일자', 'STND_DT': '결산일'
    }
    mapping.update(patch)
    
    df.columns = [c.upper() for c in df.columns]
    return df.rename(columns=mapping)

# 2. [실행] 11개 데이터셋 라벨링 (여기서 ratio_l이 정의됩니다)
print("🏷️ 11개 데이터셋 라벨링 및 변수 할당 시작...")
ov_l = smart_labeler_final(ov, 'company_overview', sc_score)
bs_l = smart_labeler_final(bs, 'company_financial_statement', sc_score)
is_l = smart_labeler_final(is_df, 'company_income_statement', sc_score)
ratio_l = smart_labeler_final(df_ratio_raw, 'company_financial_ratio', sc_score) # 핵심!
cg_l = smart_labeler_final(cg, 'company_credit_grade', sc_score)
od_l = smart_labeler_final(od, 'company_overdue', sc_score)
emp_l = smart_labeler_final(emp, 'company_employee_statistics', sc_score)
rep_l = smart_labeler_final(rep, 'company_representative_history', sc_score)
cap_l = smart_labeler_final(cap, 'company_capital_status', sc_score)
link_l = smart_labeler_final(link, 'company_links', sc_point)
bnpl_l = smart_labeler_final(bnpl, 'view_pay_bnpl_request_history', sc_point)

# 3. [병합] 중복 없는 안전한 조인
def build_ultimate_mart(ov_l, bs_l, is_l, ratio_l, cg_l, od_l, emp_l, rep_l, cap_l, link_l, bnpl_l):
    print("🚀 276홀딩스 마스터 심사 마트 병합 시작...")
    
    # 1사 1행 원칙 (최신 데이터만)
    def to_latest(df, date_col='결산일'):
        if df is None: return None
        if date_col in df.columns:
            return df.sort_values(date_col).drop_duplicates('회사 아이디', keep='last')
        return df.drop_duplicates('회사 아이디', keep='last')

    master = to_latest(ov_l, '설립일자')
    
    # 순차 병합 대상
    slates = [
        to_latest(bs_l), to_latest(is_l), to_latest(ratio_l), 
        to_latest(cg_l, '평가일자'), to_latest(emp_l, '기준일자')
    ]

    for target in slates:
        if target is not None:
            # 중복 컬럼 방지 (회사 아이디 제외)
            new_cols = target.columns.difference(master.columns).tolist() + ['회사 아이디']
            master = pd.merge(master, target[new_cols], on='회사 아이디', how='left')

    print(f"✅ 통합 완료: {len(master)}개 기업, {len(master.columns)}개 지표 확보")
    return master

# 최종 실행
df_final_mart = build_ultimate_mart(ov_l, bs_l, is_l, ratio_l, cg_l, od_l, emp_l, rep_l, cap_l, link_l, bnpl_l)

🏷️ 11개 데이터셋 라벨링 및 변수 할당 시작...
🚀 276홀딩스 마스터 심사 마트 병합 시작...
✅ 통합 완료: 1514개 기업, 535개 지표 확보


In [25]:
# [검증 1] 주요 재무비율 컬럼 존재 및 데이터 충진율(Coverage) 확인
key_ratios = ['부채비율', '영업이익율', '이자보상배수', '유동비율', '매출액증가율']

print("📊 [재무비율 데이터 통합 검증]")
print("-" * 50)
for col in key_ratios:
    if col in df_final_mart.columns:
        count = df_final_mart[col].notnull().sum()
        ratio = (count / len(df_final_mart)) * 100
        print(f"✅ {col:<10} | 데이터 존재: {count:>5}건 ({ratio:>5.1f}%)")
    else:
        print(f"❌ {col:<10} | 컬럼이 존재하지 않습니다.")

📊 [재무비율 데이터 통합 검증]
--------------------------------------------------
✅ 부채비율       | 데이터 존재:  1157건 ( 76.4%)
✅ 영업이익율      | 데이터 존재:  1141건 ( 75.4%)
✅ 이자보상배수     | 데이터 존재:  1001건 ( 66.1%)
✅ 유동비율       | 데이터 존재:  1152건 ( 76.1%)
✅ 매출액증가율     | 데이터 존재:  1131건 ( 74.7%)


In [26]:
def audit_master_database(df):
    print("🩺 [276 Audit Engine] 데이터베이스 종합 진단 보고서")
    print("=" * 60)
    
    # 1. 규모 및 중복 점검
    total_rows = len(df)
    unique_ids = df['회사 아이디'].nunique()
    print(f"📍 총 기업 수: {total_rows}개")
    print(f"📍 중복 ID 여부: {'⚠️ 위험 (중복존재)' if total_rows != unique_ids else '✅ 안전 (1사 1행)'}")
    
    # 2. 결측치 상위 10개 지표 (어디가 비어있는가?)
    print("\n⚠️ [결측치 상위 지표]")
    null_counts = df.isnull().sum().sort_values(ascending=False)
    null_ratio = (null_counts / total_rows) * 100
    display(pd.DataFrame({'결측건수': null_counts, '결측률(%)': null_ratio}).head(10))
    
    # 3. 주요 재무지표 기술 통계 (이상치 확인)
    print("\n📈 [주요 재무지표 분포 요약]")
    target_cols = ['부채비율', '영업이익율', '이자보상배수', '종업원수']
    # 존재하는 컬럼만 통계 추출
    exist_cols = [c for c in target_cols if c in df.columns]
    display(df[exist_cols].describe().T[['mean', 'min', '50%', 'max']])

# 진단 실행
audit_master_database(df_final_mart)

🩺 [276 Audit Engine] 데이터베이스 종합 진단 보고서
📍 총 기업 수: 1514개
📍 중복 ID 여부: ✅ 안전 (1사 1행)

⚠️ [결측치 상위 지표]


,결측건수,결측률(%)
주식배당금수익,1514,100.0
제품매출원가,1514,100.0
임대주택자산처분이익,1514,100.0
기타장단기유가증권처분손실,1514,100.0
임대주택자산처분손실,1514,100.0
_AB_CDC_DELETED_AT,1514,100.0
광고대행수수료,1514,100.0
기타장단기유가증권처분이익,1514,100.0
기밀비,1514,100.0
법인세효과,1514,100.0



📈 [주요 재무지표 분포 요약]


,mean,min,50%,max
영업이익율,-97.638002,-41466.50,3.06,76.50
이자보상배수,268.489780,-184907.14,1.66,435364.48
종업원수,371.636867,0.00,12.00,129524.00
